In [82]:
#imports
#python
import os, sys
import scipy
from datetime import datetime
import re
from collections import defaultdict, OrderedDict
import itertools
# image and data processing
import torch
from torchvision.models import alexnet
from torchvision import transforms as T
from PIL import Image , ImageOps
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import scipy.optimize as optimize
from scipy.linalg import block_diag
from scipy.spatial.distance import cdist
from munkres import Munkres, print_matrix
#qiskit and d-wave
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo
import dimod
from dimod import BinaryQuadraticModel, BINARY
from dwave.samplers import SimulatedAnnealingSampler
from dwave.system import DWaveSampler, EmbeddingComposite
#qubovert
import qubovert as qv
from qubovert import boolean_var, QUBO,PUBO

In [83]:
#search for and find the files
def path_joiner(image_name, root_dir = None): #recursive search for the image
    if root_dir is None:
        root_dir = os.getcwd()
    for dirpath , _, filenames in os.walk(root_dir):
        if image_name in filenames:
            return os.path.join(dirpath, image_name)
#usage
mat_files = ['Cars_000a.mat','Cars_004b.mat','Cars_006a.mat', 'Cars_008a.mat', 'Cars_013b.mat','Cars_014b.mat','Cars_015a.mat']
paths = []
for fi in mat_files:
    paths.append(path_joiner(fi))

num_views = len(mat_files)
print(num_views)

7


In [84]:
#Keypoints extraction
def keypoint(image_path, max_points=None): # extract a desired number of keypoints from the images
    keypoints1 = scipy.io.loadmat(image_path)["pts_coord"]
    if max_points is not None:
        keypoints1 = keypoints1[:, :max_points]
    return keypoints1
def keypoints_dict(image_names: list, max_points: int):
    keypoints = {}
    for image_name in image_names:
        base_name = os.path.splitext(image_name)[0]
        full_path = path_joiner(image_name)
        if full_path:
            kps = keypoint(full_path, max_points)
            keypoints[base_name] = kps
        else:
            print(f"[Warning] image not found: {image_name}")
    return keypoints
points_dic = keypoints_dict(mat_files,max_points=4)
print(f'This is the keypoints dict {points_dic}')

This is the keypoints dict {'Cars_000a': array([[ 77.63201872, 234.24966578,  20.27907754,  68.8084893 ],
       [145.18716578, 143.53275401, 102.72393048,  97.2092246 ]]), 'Cars_004b': array([[ 94.63235294, 202.20588235,  64.31617647,  81.91911765],
       [116.99448529, 116.99448529,  90.10110294,  84.72242647]]), 'Cars_006a': array([[ 47.93528064, 151.86426117,  45.965063  ,  69.60767468],
       [ 98.15864834, 121.80126002,  57.2766323 ,  54.81386025]]), 'Cars_008a': array([[ 92.57309069, 239.21867542,  51.89767303,  85.7359314 ],
       [145.1473747 , 138.18973747, 107.14797136,  88.31237411]]), 'Cars_013b': array([[ 60.57797165, 179.33478735,  55.67066521,  81.18865867],
       [114.96728462, 151.77208288,  86.99563795,  79.6346783 ]]), 'Cars_014b': array([[ 79.83608491, 220.51176453,  57.61910377,  70.94929245],
       [116.73643868, 112.30714417,  77.8567217 ,  64.52653302]]), 'Cars_015a': array([[ 57.82954545, 244.20227273,  14.04318182,  58.33217274],
       [153.42613636, 15

In [ ]:
#Feature extraction using AlexNet

def alexnet_feature_extractor(layer= 'conv4'):
    model = alexnet(pretrained=True)
    model.eval()
    if layer == 'conv4':
        return torch.nn.Sequential(*list(model.features)[:10])
    elif layer == 'conv5':
        return torch.nn.Sequential(*list(model.features)[:12])
    else:
        raise ValueError("Invalid layer. Choose 'conv4' or 'conv5'.")
def extract_features(keypoints_dict, patch_size=80, layer='conv4'):
    feature_extractor = alexnet_feature_extractor(layer)
    transform = T.Compose([
        T.Resize((227, 227)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor.to(device)
    features = {}

    for image_name, keypoints in keypoints_dict.items():
        img_path = path_joiner(image_name + '.png')
        img = Image.open(img_path).convert('RGB')
        # img = Image.open(img_path).convert('L')
        img_np = np.array(img)
        feature_list = []

        for (x, y) in keypoints.T:
            half = patch_size // 2

            # Pad the image if keypoint is near the edge
            padding = (half, half, half, half)  # left, top, right, bottom
            padded_img = ImageOps.expand(img, border=padding, fill=0)

            # shift keypoint for padding the edges
            x_padded = x + half
            y_padded = y + half

            # safely crop the full patch_size
            patch = padded_img.crop((x_padded - half, y_padded - half,
                                    x_padded + half, y_padded + half))
            # patch = patch.convert('RGB')  # duplicate L channel into R, G, B for alexnet
            patch_tensor = transform(patch).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = feature_extractor(patch_tensor)
                feat = feat.mean(dim=[2, 3]) # to flatten the output dimensions
            feature_list.append(feat.squeeze().cpu().numpy())

        features[image_name] = np.stack(feature_list)
    return features
features = extract_features(points_dic)
print(features)

d:\Programs\anaconda3\envs\quantum-sync\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Programs\anaconda3\envs\quantum-sync\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


{'Cars_000a': array([[0.00705218, 0.        , 0.04452105, ..., 0.0139769 , 0.21914941,
        0.03882647],
       [0.03548701, 0.        , 0.11592701, ..., 0.14416793, 0.05984103,
        0.23470877],
       [0.00176703, 0.06730222, 0.03098544, ..., 0.03092694, 0.0477472 ,
        0.06404933],
       [0.04276872, 0.00080363, 0.11364744, ..., 0.14013734, 0.08242278,
        0.2035586 ]], dtype=float32), 'Cars_004b': array([[0.66649663, 0.12129387, 0.08973002, ..., 0.10984258, 0.10632853,
        0.00806867],
       [0.03402828, 0.0998574 , 0.00263216, ..., 0.105927  , 0.0158512 ,
        0.12270136],
       [0.2781731 , 0.43241793, 0.11791825, ..., 0.13457423, 0.08541029,
        0.24328254],
       [0.2771119 , 0.6368087 , 0.09898125, ..., 0.0952096 , 0.04141085,
        0.13019522]], dtype=float32), 'Cars_006a': array([[2.9633093e+00, 8.8076144e-02, 1.7436068e-01, ..., 4.6850225e-01,
        2.5575873e-01, 1.8673512e-01],
       [1.9903305e+00, 9.7714417e-02, 1.0024991e-01, ..., 3.43

In [86]:
#features have been extracted, now we need to calculate the distance matrix
def pairwise_permutations(features_dict, pm_size: int) -> dict: # compute the P_ij
    P = {}
    image_list = list(features_dict.keys())

    for i in range(len(image_list)):
        key0= f"P{i+1}{i+1}"
        P[key0] = np.eye(pm_size)
        for j in range(i + 1, len(image_list)):
            img1 = image_list[i]
            img2 = image_list[j]

            feats1 = features_dict[img1]
            feats2 = features_dict[img2]
            similarity = cosine_similarity(feats1, feats2) # equivalent to cosine similarity, since the features are normalized
            print("this is the similarity matrix")
            print(similarity)
            cost_mat = (1 - similarity)
            row_ind, col_ind = linear_sum_assignment(cost_mat)
            n = len(row_ind)
            perm_matrix = np.zeros((n, n), dtype=int)
            perm_matrix[row_ind, col_ind] = 1
            key1 = f"P{i+1}{j+1}"
            P[key1] = perm_matrix
            key2 = f"P{j+1}{i+1}"
            P[key2] = perm_matrix.T
    return P
P = pairwise_permutations(features, 4)
print(len(P))
print(P)

this is the similarity matrix
[[0.69577205 0.6658493  0.62142533 0.6358284 ]
 [0.6823563  0.64909583 0.6126669  0.6215356 ]
 [0.6988735  0.6410619  0.62297446 0.6373588 ]
 [0.7323671  0.6873504  0.672171   0.66983277]]
this is the similarity matrix
[[0.46169263 0.6744133  0.45100632 0.3872033 ]
 [0.4315078  0.67653036 0.43977988 0.3807143 ]
 [0.46977082 0.54163736 0.49604562 0.44549772]
 [0.49628103 0.67305624 0.47794876 0.42369574]]
this is the similarity matrix
[[0.7134602  0.72130054 0.5691468  0.5204465 ]
 [0.6889612  0.69538885 0.5450427  0.52558225]
 [0.5207775  0.5197867  0.60309935 0.5639893 ]
 [0.61809397 0.6138096  0.6401099  0.61051404]]
this is the similarity matrix
[[0.6987931  0.7066508  0.6501217  0.58218986]
 [0.67420346 0.7617438  0.61707246 0.5471699 ]
 [0.6795943  0.549256   0.65250707 0.63683456]
 [0.75591815 0.66857356 0.6982304  0.6692317 ]]
this is the similarity matrix
[[0.6011635  0.6339751  0.7491724  0.683646  ]
 [0.5509828  0.5789692  0.703477   0.6626184 ]


In [59]:
def qubo_form_maker(P: dict,
                   num_views: int,
                   num_keypoints: int,
                   penalty: float = 1.5) -> QUBO:
    m = num_views
    n = num_keypoints
    N = n * n
    #constructing the Q_prime (m.N)x(m.N)
    Q_prime = np.zeros((m*N, m*N))
    for key, p_mat_ij in P.items():
        # extract i and j from key like "P12"
        i = int(key[1]) - 1  # subtract 1 for zero-based indexing
        j = int(key[2]) - 1
        block_ij = np.kron(np.eye(n), p_mat_ij)
        for u in range(N):
            for v in range(N):
                Q_prime[i*N + u, j*N + v] -= block_ij[u, v]
    # constructing the A and b as mentioned in the paper
    A = []
    b = []

    for i in range(m):
        #Row constraints
        for row in range(n):
            constraint = np.zeros(m*N)
            for col in range(n):
                constraint[i*N+row*n+col] = 1
            A.append(constraint)
            b.append(1)
        # Column constraint
        for col in range(n):
            constraint = np.zeros(m*N)
            for row in range(n):
                idx = i * N + row * n + col
                constraint[idx] = 1
            A.append(constraint)
            b.append(1)
    A = np.array(A)
    b = np.array(b)
    #constructing Q and s
    Q = Q_prime + penalty * A.T @ A
    s = -2 * penalty * A.T @ b
    # putting things together
    return Q,s

In [87]:
def qubo_form_maker_fixed_gauge(P: dict,
                                num_views: int,
                                num_keypoints: int,
                                penalty: float = 2.5) -> tuple[np.ndarray, np.ndarray]:
    """
    Constructs the QUBO matrix Q and linear vector s for permutation synchronization,
    fixing the gauge by setting the first absolute permutation matrix X1 = I.

    Args:
        P: Dictionary of pairwise permutations Pij, e.g., P['P12'], P['P21'].
           Keys are 1-based indexed, e.g., "P12", "P23".
        num_views: Total number of views (m).
        num_keypoints: Number of keypoints per view (n).
        penalty: Lambda value for the constraint penalty term.

    Returns:
        A tuple containing the QUBO matrix Q and linear vector s for the
        optimization variables corresponding to X2, ..., Xm.
    """
    m = num_views
    n = num_keypoints
    N = n * n # Number of variables per view matrix (n^2)

    if m <= 1:
        # Nothing to optimize if there's only one view (or zero)
        return np.array([[]]), np.array([])

    m_opt = m - 1 # Number of views to optimize (X2 to Xm)
    num_vars = m_opt * N # Total number of binary variables in the QUBO

    # --- Initialize QUBO components for optimization variables ---
    Q_prime_opt = np.zeros((num_vars, num_vars))
    s_opt_initial = np.zeros(num_vars) # Linear terms arising from fixing X1=I

    # --- Construct the constant vector for vec(X1) = vec(I) ---
    x1 = np.zeros(N)
    for k in range(n):
        x1[k * n + k] = 1

    # --- Populate Q_prime_opt and s_opt_initial from the objective function ---
    # Objective: sum_{i,j} ||Pij - Xi Xj^T||^2 = const - 2 * sum_{i,j} tr(Xj X_i^T Pij)
    # We want to minimize: - sum_{i,j} vec(Xi)^T (I kron Pij) vec(Xj)

    for key, p_mat_ij in P.items():
        # Extract i and j (0-based index for internal calculations)
        # Assumes keys are like 'P12', 'P21', etc. (1-based)
        try:
            i = int(key[1]) - 1
            j = int(key[2]) - 1
        except (IndexError, ValueError):
            print(f"Warning: Could not parse view indices from key '{key}'. Skipping.")
            continue

        # Calculate the block matrix for the interaction term
        block_ij = np.kron(np.eye(n), p_mat_ij) # This is (I kron Pij)

        if i > 0 and j > 0:
            # Interaction between Xi and Xj (where i, j != 1)
            # Adjust indices for the optimization variables (X2...Xm map to indices 0...m_opt-1)
            opt_i_idx = i - 1
            opt_j_idx = j - 1
            start_row = opt_i_idx * N
            end_row = start_row + N
            start_col = opt_j_idx * N
            end_col = start_col + N
            # Subtract term based on -vec(Xi)^T * block_ij * vec(Xj)
            Q_prime_opt[start_row:end_row, start_col:end_col] -= block_ij

        elif i == 0 and j > 0:
            # Interaction between X1 (Identity) and Xj
            # Term is -vec(X1)^T * block_1j * vec(Xj) -> contributes to linear part of Xj
            opt_j_idx = j - 1
            linear_contrib = -x1.T @ block_ij # This is a row vector, needs reshaping? No, result is scalar * vec(Xj) coefs.
            start_col = opt_j_idx * N
            end_col = start_col + N
            s_opt_initial[start_col:end_col] += linear_contrib

        elif i > 0 and j == 0:
            # Interaction between Xi and X1 (Identity)
            # Term is -vec(Xi)^T * block_i1 * vec(X1) -> contributes to linear part of Xi
            opt_i_idx = i - 1
            # Need vec(Xi)^T @ block_i1 @ x1 = (block_i1 @ x1)^T @ vec(Xi)
            linear_contrib = -(block_ij @ x1) # block_ij here is (I kron Pi1)
            start_row = opt_i_idx * N
            end_row = start_row + N
            s_opt_initial[start_row:end_row] += linear_contrib

        # Case i=0 and j=0 corresponds to tr(X1 X1^T P11) = tr(I I I) = n, a constant, ignore.

    # --- Construct the constraint matrix A_opt and vector b_opt ---
    # Constraints only apply to X2, ..., Xm
    num_constraints = m_opt * 2 * n # Each of m_opt matrices has n row + n col constraints
    A_opt = np.zeros((num_constraints, num_vars))
    b_opt = np.ones(num_constraints) # All constraints require sums to be 1

    current_constraint_row = 0
    for view_idx_opt in range(m_opt): # Iterate through X2...Xm (indexed 0 to m_opt-1)
        view_offset = view_idx_opt * N

        # Row sum constraints for current view
        for row in range(n):
            for col in range(n):
                var_index = view_offset + row * n + col
                A_opt[current_constraint_row, var_index] = 1
            current_constraint_row += 1

        # Column sum constraints for current view
        for col in range(n):
            for row in range(n):
                var_index = view_offset + row * n + col
                A_opt[current_constraint_row, var_index] = 1
            current_constraint_row += 1

    # --- Calculate final Q and s ---
    Q = Q_prime_opt + penalty * (A_opt.T @ A_opt)
    s = s_opt_initial - 2 * penalty * (A_opt.T @ b_opt)

    return Q, s

In [88]:
num_keypoints = 4
Q,s = qubo_form_maker_fixed_gauge(P, num_views, num_keypoints, 5.0)
print(Q)
print(s)


[[9. 5. 5. ... 0. 0. 0.]
 [5. 9. 5. ... 0. 0. 0.]
 [5. 5. 9. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 9. 5. 5.]
 [0. 0. 0. ... 5. 9. 5.]
 [0. 0. 0. ... 5. 5. 9.]]
[-20. -22. -20. -20. -20. -20. -20. -22. -22. -20. -20. -20. -20. -20.
 -22. -20. -20. -20. -22. -20. -20. -22. -20. -20. -20. -20. -20. -22.
 -22. -20. -20. -20. -20. -22. -20. -20. -22. -20. -20. -20. -20. -20.
 -22. -20. -20. -20. -20. -22. -20. -20. -22. -20. -20. -22. -20. -20.
 -20. -20. -20. -22. -22. -20. -20. -20. -20. -22. -20. -20. -20. -20.
 -22. -20. -22. -20. -20. -20. -20. -20. -20. -22. -20. -22. -20. -20.
 -22. -20. -20. -20. -20. -20. -22. -20. -20. -20. -20. -22.]


In [89]:
def dwave_sim_anneal(Q,s, num_trials: int = 100):
    # vec_len=len(s)
    # linear = { f"x_{k}": Q[k,k] + s[k] for k in range(vec_len) }
    # quadratic = {
    #     (f"x_{i}", f"x_{j}"): 2.0*Q[i,j]
    #     for i in range(vec_len) for j in range(i+1, vec_len)
    #     if abs(Q[i,j]) > 1e-8
    # }
    
    sampler = SimulatedAnnealingSampler()
    bqm = BinaryQuadraticModel(s, Q,0.0, vartype=BINARY)
    res = sampler.sample(bqm,
        num_reads = num_trials
    )
    best_sample = res.first.sample
    best_energy = res.first.energy
    return best_sample,best_energy, res
best_sample, best_energy, sim_res = dwave_sim_anneal(Q,s,100000)
print(best_sample)
print(best_energy)

{0: np.int8(0), 1: np.int8(1), 2: np.int8(0), 3: np.int8(0), 4: np.int8(0), 5: np.int8(0), 6: np.int8(0), 7: np.int8(1), 8: np.int8(1), 9: np.int8(0), 10: np.int8(0), 11: np.int8(0), 12: np.int8(0), 13: np.int8(0), 14: np.int8(1), 15: np.int8(0), 16: np.int8(0), 17: np.int8(1), 18: np.int8(0), 19: np.int8(0), 20: np.int8(0), 21: np.int8(0), 22: np.int8(1), 23: np.int8(0), 24: np.int8(1), 25: np.int8(0), 26: np.int8(0), 27: np.int8(0), 28: np.int8(0), 29: np.int8(0), 30: np.int8(0), 31: np.int8(1), 32: np.int8(0), 33: np.int8(1), 34: np.int8(0), 35: np.int8(0), 36: np.int8(1), 37: np.int8(0), 38: np.int8(0), 39: np.int8(0), 40: np.int8(0), 41: np.int8(0), 42: np.int8(1), 43: np.int8(0), 44: np.int8(0), 45: np.int8(0), 46: np.int8(0), 47: np.int8(1), 48: np.int8(0), 49: np.int8(1), 50: np.int8(0), 51: np.int8(0), 52: np.int8(0), 53: np.int8(0), 54: np.int8(1), 55: np.int8(0), 56: np.int8(1), 57: np.int8(0), 58: np.int8(0), 59: np.int8(0), 60: np.int8(0), 61: np.int8(0), 62: np.int8(0), 6

In [90]:
def extract_permutation_matrices(best_sample: dict, num_views: int, num_keypoints: int):
    matrices = {}
    n = num_keypoints
    n2 = n ** 2

    

    for v in range(0,num_views):
        X = np.zeros((n, n), dtype=int)
        
        for i in range(n):
            for j in range(n):
                var_idx = i * n + j
                var_name = var_idx
                X[i, j] = best_sample.get(var_name, 0)
        matrices[f'X{v+1}'] = X

    return matrices


In [91]:
def extract_permutation_matrices_fixed_gauge(best_sample: dict, num_views: int, num_keypoints: int):
    """
    Extracts permutation matrices from the QUBO result, accounting for X1=I gauge fixing.

    Args:
        best_sample: Dictionary representing the lowest energy sample from the solver.
                     Keys should correspond to the variable indices (0 to (m-1)*n^2 - 1).
        num_views: Total number of views (m).
        num_keypoints: Number of keypoints per view (n).

    Returns:
        A dictionary containing the absolute permutation matrices X1, X2, ..., Xm.
        X1 is always the identity matrix.
    """
    matrices = {}
    n = num_keypoints
    N = n * n # n^2
    m_opt = num_views - 1

    # Set X1 = Identity
    matrices['X1'] = np.eye(n, dtype=int)

    # Extract X2, ..., Xm from the solution vector
    # The keys in best_sample should ideally be integers 0, 1, 2... representing the variable index
    # If keys are strings like 'x_0', 'x_1', we need to parse them. Assuming integer keys for simplicity.

    solution_vector = np.zeros(m_opt * N)
    for idx, val in best_sample.items():
         # Handle potential string keys from some samplers
        try:
            int_idx = int(str(idx).split('_')[-1]) # Attempt to parse keys like 'x_0', 'x_1' or just 0, 1
        except ValueError:
             try:
                 int_idx = int(idx) # Handle integer keys
             except ValueError:
                 print(f"Warning: Could not parse variable index from key '{idx}'. Skipping.")
                 continue

        if 0 <= int_idx < len(solution_vector):
             solution_vector[int_idx] = val
        else:
             print(f"Warning: Index {int_idx} out of bounds for solution vector size {len(solution_vector)}. Skipping.")


    for v_opt in range(m_opt): # v_opt runs from 0 to m_opt-1 (representing views 2 to m)
        X = np.zeros((n, n), dtype=int)
        view_offset = v_opt * N
        current_view_vars = solution_vector[view_offset : view_offset + N]

        for i in range(n):
            for j in range(n):
                var_idx_in_view = i * n + j
                if var_idx_in_view < len(current_view_vars):
                     X[i, j] = int(current_view_vars[var_idx_in_view])
                else:
                     # This should not happen if slicing is correct
                     print(f"Warning: Index out of bounds when reconstructing matrix X{v_opt+2}")


        matrices[f'X{v_opt + 2}'] = X # Store as X2, X3, ...

    return matrices
#
matrices = extract_permutation_matrices_fixed_gauge(best_sample, num_views, num_keypoints)
print(matrices)

{'X1': array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'X2': array([[0, 1, 0, 0],
       [0, 0, 0, 1],
       [1, 0, 0, 0],
       [0, 0, 1, 0]]), 'X3': array([[0, 1, 0, 0],
       [0, 0, 1, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 1]]), 'X4': array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'X5': array([[0, 1, 0, 0],
       [0, 0, 1, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 1]]), 'X6': array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]), 'X7': array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]])}


In [92]:
def is_valid_permutation_matrix(X): #check the validity of the absolute permutation matrices(all rows and columns sum to 1)
    return (
        np.all(np.sum(X, axis=0) == 1) and
        np.all(np.sum(X, axis=1) == 1) and
        np.all((X == 0) | (X == 1))
    )

for k, i in matrices.items():
    print(f'matrix {k} is valid : {is_valid_permutation_matrix(i)}')

matrix X1 is valid : True
matrix X2 is valid : True
matrix X3 is valid : True
matrix X4 is valid : True
matrix X5 is valid : True
matrix X6 is valid : True
matrix X7 is valid : True


In [93]:
import os
from datetime import datetime
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import matplotlib.lines as mlines
import random
import string

# Helper to generate visually distinct colors
def generate_n_colors(n):
    cmap = plt.cm.get_cmap("tab10", n)
    return [cmap(i) for i in range(n)]

# Main visualization and result-saving function
def visualize_and_save_result(points_list, permutation_matrices, mat_files, best_energy, penalty_value, output_dir="./results"):
    os.makedirs(output_dir, exist_ok=True)

    num_views = len(points_list)
    num_keypoints = points_list[0].shape[1]

    # Construct matched keypoints indices across views using the permutation matrices
    # X1 = identity, we use its keypoint indices as the base
    matches = [[] for _ in range(num_keypoints)]  # each inner list will contain 1 index per image

    for i in range(num_keypoints):
        matches[i].append(i)  # X1 is identity

    for view_idx in range(1, num_views):
        X = permutation_matrices[f'X{view_idx+1}']
        for i in range(num_keypoints):
            target = np.argmax(X[i])
            matches[i].append(target)

    # Load images
    images = []
    for mat_file in mat_files:
        img_path = path_joiner(mat_file.replace('.mat', '.png'))  # assumes .png format
        images.append(Image.open(img_path).convert("RGB"))

    # Create canvas for side-by-side image
    widths, heights = zip(*(img.size for img in images))
    total_width = sum(widths)
    max_height = max(heights)

    canvas = Image.new('RGB', (total_width, max_height), color=(255, 255, 255))
    x_offsets = []
    x_offset = 0
    for img in images:
        canvas.paste(img, (x_offset, 0))
        x_offsets.append(x_offset)
        x_offset += img.size[0]

    # Plot keypoints
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.imshow(canvas)
    ax.axis('off')

    colors = generate_n_colors(num_keypoints)
    radius = 5

    for keypoint_idx, match in enumerate(matches):
        color = colors[keypoint_idx]
        prev_x, prev_y = None, None
        for view_idx, kp_idx in enumerate(match):
            pt = points_list[view_idx][:, kp_idx]
            x = x_offsets[view_idx] + pt[0]
            y = pt[1]
            circle = plt.Circle((x, y), radius, color=color, fill=True)
            ax.add_patch(circle)
            if prev_x is not None:
                line = mlines.Line2D([prev_x, x], [prev_y, y], color=color, linewidth=2)
                ax.add_line(line)
            prev_x, prev_y = x, y

    # Save image and text file
    short_names = "_".join([os.path.splitext(f)[0].lower() for f in mat_files])
    timestamp = datetime.now().strftime("%H%M%S")
    base_filename = f"{short_names}_{timestamp}"
    image_save_path = os.path.join(output_dir, base_filename + ".png")
    txt_save_path = os.path.join(output_dir, base_filename + ".txt")

    fig.savefig(image_save_path, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)

    with open(txt_save_path, "w") as f:
        f.write(f"Images: {mat_files}\n")
        f.write(f"Penalty: {penalty_value}\n")
        f.write(f"Energy: {best_energy}\n")
        f.write(f"Permutation Matrices:\n")
        for k, mat in permutation_matrices.items():
            f.write(f"{k}:\n{mat}\n\n")

    return image_save_path, txt_save_path

In [94]:
def keypoints_list(image_paths: list, max_points: int):
    keypoints = []
    for image_path in image_paths:
        keypoints1 = keypoint(image_path, max_points)
        keypoints.append(keypoints1)
    return keypoints
points_list = keypoints_list(paths,4)

In [95]:
#

image_path, txt_path = visualize_and_save_result(
    points_list=points_list,
    permutation_matrices=matrices,
    mat_files=mat_files,
    best_energy=best_energy,
    penalty_value=1.5  
)
print(f"Saved image to: {image_path}")
print(f"Saved text to: {txt_path}")

Saved image to: ./results\cars_000a_cars_004b_cars_006a_cars_008a_cars_013b_cars_014b_cars_015a_132153.png
Saved text to: ./results\cars_000a_cars_004b_cars_006a_cars_008a_cars_013b_cars_014b_cars_015a_132153.txt


C:\Users\sasan\AppData\Local\Temp\ipykernel_9176\2230953731.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("tab10", n)
